# Adaptive Multiscale Spectro-Topological (AMST) Shape Descriptor
## Version 10 — Four Critical Fixes Applied

**Fixes over v9:**
- **FIX 1 (Critical):** Fair comparison — ALL methods now evaluated with the same SVM-RBF classifier.  
  Stacking ensemble reported as a *separate additional experiment*, not mixed into the baseline table.
- **FIX 2 (Critical):** Y-axis in Figures 8, 9, 13 now correctly shows 0–100% (was 0–1 raw fraction).
- **FIX 3 (Minor):** Dimension text unified everywhere — 466 dims throughout (C1=144, C2=90, C3=210, C5=22).
- **FIX 4 (Minor):** Ablation uses SVM for every sub-variant including Full AMST, then separately shows  
  the stacking gain so classifier and descriptor contributions are not conflated.

**Unchanged from v9:**
- Per-fold attention fitting (no data leakage)
- All 7 baselines, 5-fold stratified CV, paired t-test, seed=42
- 13 figures, noise/occlusion/rotation/retrieval experiments

In [ ]:
# ── Cell 1: Install Dependencies ───────────────────────────────────────────────
!pip install -q PyWavelets ripser persim xgboost
!pip install -q scikit-image scikit-learn matplotlib seaborn scipy numpy pandas tqdm gdown

import os, sys, warnings, json, glob
warnings.filterwarnings('ignore')
print('All packages installed.')

In [ ]:
# ── Cell 2: All Imports & Reproducibility ───────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import copy, re, warnings
warnings.filterwarnings('ignore')
import scipy
import scipy.stats
import scipy.special
from scipy import ndimage
from scipy.interpolate import interp1d

import pywt
print(f'PyWavelets OK: {pywt.__version__}')

try:
    from ripser import ripser
    from persim import plot_diagrams
    RIPSER_AVAILABLE = True
    print('Ripser + Persim: OK')
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'ripser', 'persim'], check=True)
    from ripser import ripser
    from persim import plot_diagrams
    RIPSER_AVAILABLE = True
    print('Ripser + Persim installed.')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost OK: {xgb.__version__}')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not available — stacking will use RF-only.')

from skimage import io, color, transform, feature, measure
from skimage import img_as_float
from skimage.filters import threshold_otsu
from skimage.morphology import binary_closing, binary_opening, disk, remove_small_objects
from skimage.measure import find_contours
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score)

SEED = 42
AMST_DIM = 466  # FIX 3: single source of truth for AMST dimension
np.random.seed(SEED)
print(f'NumPy {np.__version__} | SciPy {scipy.__version__} | PyWavelets {pywt.__version__}')
print(f'Seed={SEED} | Ripser={RIPSER_AVAILABLE} | XGBoost={XGB_AVAILABLE} | AMST_DIM={AMST_DIM}')

In [ ]:
# ── Cell 3: Download KIMIA-216 ──────────────────────────────────────────
import gdown, zipfile, shutil

DATA_DIR = Path('/content/kimia216')
DATA_DIR.mkdir(exist_ok=True)
EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.gif'}

manual_zip = Path('/content/Kimia216-Original.zip')
if manual_zip.exists():
    print('Found manually uploaded zip. Extracting...')
    with zipfile.ZipFile(str(manual_zip), 'r') as z:
        z.extractall('/content/kimia216')
    manual_zip.unlink()
    print('Manual zip extraction complete.')

existing = [f for f in DATA_DIR.rglob('*') if f.suffix.lower() in EXT]
if len(existing) < 200:
    print('Downloading KIMIA-216...')
    url = 'https://drive.google.com/uc?id=1xGamTIPywVJIwkBdpLH4BHDMQ8aBq4D7'
    zip_path = '/content/kimia216.zip'
    try:
        gdown.download(url, zip_path, quiet=False)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall('/content/kimia216')
        print('Download complete.')
    except Exception as e:
        print(f'gdown failed: {e}')
        os.system('wget -q "https://github.com/nschmidtg/KIMIA216/archive/refs/heads/master.zip" -O /content/kimia_gh.zip')
        with zipfile.ZipFile('/content/kimia_gh.zip', 'r') as z:
            z.extractall('/content/kimia_gh')
        for f in Path('/content/kimia_gh').rglob('*'):
            if f.suffix.lower() in EXT:
                shutil.copy(str(f), str(DATA_DIR / f.name))
        print('GitHub fallback complete.')
else:
    print(f'Already have {len(existing)} images — skipping download.')

image_files = sorted([f for f in DATA_DIR.rglob('*') if f.suffix.lower() in EXT])
print(f'Total image files found: {len(image_files)}')
for p in image_files[:6]:
    print(f'  {p}')

In [ ]:
# ── Cell 4: Parse Labels ───────────────────────────────────────
def parse_kimia_label(filepath):
    name = Path(filepath).stem.lower()
    label = re.sub(r'[-_]?\d+$', '', name).strip('-_')
    return label if label else name

samples    = [(img, parse_kimia_label(img)) for img in image_files]
labels_raw = [s[1] for s in samples]
unique_classes = sorted(set(labels_raw))
print(f'Samples: {len(samples)} | Classes: {len(unique_classes)}')
vc = pd.Series(labels_raw).value_counts().sort_index()
print('Class distribution:')
print(vc.to_dict())

In [ ]:
# ── Cell 5: Image Loading & Preprocessing ─────────────────────────────────
IMG_SIZE       = (128, 128)
CONTOUR_POINTS = 256

def load_and_binarize(path, img_size=IMG_SIZE):
    img = io.imread(str(path))
    if img.ndim == 3 and img.shape[2] == 4:
        gray = color.rgb2gray(img[..., :3])
    elif img.ndim == 3:
        gray = color.rgb2gray(img)
    else:
        gray = img_as_float(img)
    gray = transform.resize(gray, img_size, anti_aliasing=True)
    try:    thresh = threshold_otsu(gray)
    except: thresh = 0.5
    binary = gray < thresh
    if binary.sum() < img_size[0] * img_size[1] * 0.02:
        binary = ~binary
    binary = binary_closing(binary, disk(3))
    binary = binary_opening(binary, disk(2))
    binary = remove_small_objects(binary.astype(bool), min_size=100)
    return binary.astype(np.uint8)

def extract_contour(binary, n_points=CONTOUR_POINTS):
    contours_list = find_contours(binary.astype(float), 0.5)
    if not contours_list:
        return np.zeros((n_points, 2))
    contour = max(contours_list, key=len)
    diffs = np.diff(contour, axis=0)
    arc   = np.r_[0, np.cumsum(np.sqrt((diffs**2).sum(axis=1)))]
    if arc[-1] < 1e-8:
        return np.zeros((n_points, 2))
    u = np.linspace(0, arc[-1], n_points, endpoint=False)
    return np.column_stack([np.interp(u, arc, contour[:, 0]),
                            np.interp(u, arc, contour[:, 1])])

def center_and_scale(c):
    c = c - c.mean(axis=0)
    r = np.sqrt((c**2).sum(axis=1)).max()
    return c / r if r > 1e-8 else c

def compute_curvature(contour):
    x, y = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x);  y1 = np.gradient(y)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    return (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5

print('Loading KIMIA-216...')
binaries, contours, curvatures, all_labels = [], [], [], []
for path, label in tqdm(samples, desc='Loading'):
    try:
        bimg = load_and_binarize(path)
        cnt  = center_and_scale(extract_contour(bimg))
        kap  = compute_curvature(cnt)
        binaries.append(bimg); contours.append(cnt)
        curvatures.append(kap); all_labels.append(label)
    except Exception as e:
        print(f'Failed: {path} — {e}')

le = LabelEncoder()
y  = le.fit_transform(all_labels)
le.classes_ = np.array([str(c) for c in le.classes_])
n_classes   = len(le.classes_)
print(f'Loaded: {len(contours)} | Classes: {n_classes}')
print(f'Class names: {list(le.classes_)}')

In [ ]:
# ── Figure 1: KIMIA-216 Dataset Samples ─────────────────────────────────
n_cols = 6
n_rows = (n_classes + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 3))
axes = axes.flatten()
fig.suptitle('Figure 1: KIMIA-216 Benchmark Dataset\n'
             f'18 Classes x 12 Instances = 216 Binary Silhouettes (128x128)',
             fontsize=14, fontweight='bold', y=1.01)
for cls_idx in range(n_classes):
    idx = np.where(y == cls_idx)[0][0]
    axes[cls_idx].imshow(binaries[idx], cmap='gray')
    axes[cls_idx].set_title(le.classes_[cls_idx].capitalize(), fontsize=9, fontweight='bold')
    axes[cls_idx].axis('off')
for ax in axes[n_classes:]:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/fig1_kimia216_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Figure 2: Shape Preprocessing Pipeline ─────────────────────────────
display_cls = [c for c in [0, 3, 7, 12] if c < n_classes][:4]
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Figure 2: Shape Preprocessing Pipeline\n'
             'Row 1: Binary | Row 2: Normalised Contour | Row 3: Curvature kappa(t)',
             fontsize=12, fontweight='bold', y=1.01)
for col, cls in enumerate(display_cls):
    idx = np.where(y == cls)[0][0]
    cls_name = le.classes_[cls].capitalize()
    axes[0, col].imshow(binaries[idx], cmap='gray')
    axes[0, col].set_title(cls_name, fontsize=10, fontweight='bold')
    axes[0, col].axis('off')
    cnt = contours[idx]
    axes[1, col].plot(cnt[:, 1], -cnt[:, 0], 'b-', lw=1.5)
    axes[1, col].plot(cnt[0, 1], -cnt[0, 0], 'ro', ms=5)
    axes[1, col].set_aspect('equal'); axes[1, col].axis('off')
    axes[1, col].set_title(f'Contour (N={CONTOUR_POINTS})', fontsize=8)
    kap = curvatures[idx]
    t = np.linspace(0, 1, len(kap))
    axes[2, col].plot(t, kap, 'g-', lw=1.2)
    axes[2, col].fill_between(t, kap, alpha=0.25, color='green')
    axes[2, col].axhline(0, color='k', lw=0.5, ls='--')
    axes[2, col].set_xlabel('t in [0,1]', fontsize=8)
    axes[2, col].set_title('Curvature kappa(t)', fontsize=8)
    axes[2, col].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/fig2_preprocessing_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Cell 6: Baseline Shape Descriptors ───────────────────────────────────
# HOG: 576-dim (128x128, cells_per_block=(1,1))
# CSS: 10 sigma scales → 20-dim
# Shape Context: 5 radial x 12 angular = 60-dim

def fourier_descriptor(contour, n_coeff=64):
    r = np.sqrt((contour**2).sum(axis=1))
    F = np.fft.fft(r); mag = np.abs(F)
    denom = mag[1] if mag[1] > 1e-8 else mag.max() + 1e-12
    mag_n = mag / denom
    return np.concatenate([mag_n[1:n_coeff + 1][:-1], np.angle(F)[1:17]])

def wavelet_descriptor(contour):
    r = np.sqrt((contour**2).sum(axis=1))
    r = r - r.mean()
    feats = []
    for wv_name in ['db4', 'haar', 'sym4']:
        safe_level = max(1, min(5, pywt.dwt_max_level(len(r), wv_name)))
        coeffs = pywt.wavedec(r, wv_name, level=safe_level, mode='periodization')
        energies = np.array([np.sum(c**2) for c in coeffs])
        feats.append(energies / (energies.sum() + 1e-12))
    max_len = max(len(f) for f in feats)
    return np.concatenate([np.pad(f, (0, max_len - len(f))) for f in feats])

def simple_hybrid_descriptor(contour):
    return np.concatenate([fourier_descriptor(contour, 64), wavelet_descriptor(contour)])

def zernike_moments(binary, max_order=10):
    h, w = binary.shape
    yg, xg = np.mgrid[-1:1:1j*h, -1:1:1j*w]
    r = np.sqrt(xg**2 + yg**2); theta = np.arctan2(yg, xg)
    mask = (r <= 1.) & (binary > 0); moments = []
    for n in range(max_order + 1):
        for m in range(-n, n + 1, 2):
            if (n - abs(m)) % 2 != 0: continue
            R = np.zeros_like(r)
            for s in range((n - abs(m)) // 2 + 1):
                c = ((-1)**s * scipy.special.factorial(n - s)) / (
                    scipy.special.factorial(s) *
                    scipy.special.factorial((n + abs(m)) // 2 - s) *
                    scipy.special.factorial((n - abs(m)) // 2 - s) + 1e-300)
                R += c * r**(n - 2*s)
            V = R * np.exp(-1j * m * theta)
            moments.append(np.abs(np.sum(V[mask] * binary[mask]) * (n + 1) / np.pi))
    return np.array(moments[:36])

def shape_context(contour, n_r=5, n_theta=12):
    N = len(contour); step = max(1, N // 64); pts = contour[::step]; n = len(pts)
    dx = pts[:, 1:2] - pts[np.newaxis, :, 1]; dy = pts[:, 0:1] - pts[np.newaxis, :, 0]
    dist = np.sqrt(dx**2 + dy**2 + 1e-12); angles = np.arctan2(dy, dx)
    log_dist = np.log(dist / (dist.max() + 1e-12) + 1e-12)
    r_bins = np.linspace(log_dist.min() - 0.01, 0.01, n_r + 1)
    t_bins = np.linspace(-np.pi, np.pi, n_theta + 1)
    H_g = np.zeros(n_r * n_theta)
    for i in range(n):
        mi = np.arange(n) != i
        H, _, _ = np.histogram2d(log_dist[i, mi], angles[i, mi], bins=[r_bins, t_bins])
        H_g += H.flatten()
    return H_g / (H_g.sum() + 1e-12)

def curvature_scale_space(contour, sigmas=None):
    if sigmas is None:
        sigmas = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512]
    x, yc = contour[:, 1], contour[:, 0]; feats = []
    for sigma in sigmas:
        xs = ndimage.gaussian_filter1d(x, sigma, mode='wrap')
        ys = ndimage.gaussian_filter1d(yc, sigma, mode='wrap')
        x1 = np.gradient(xs); x2 = np.gradient(x1)
        y1 = np.gradient(ys); y2 = np.gradient(y1)
        k = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
        feats += [float(np.sum(np.diff(np.sign(k)) != 0)), np.mean(np.abs(k))]
    return np.array(feats)

def hog_descriptor(binary):
    return feature.hog(binary.astype(np.float32), orientations=9,
                       pixels_per_cell=(16, 16), cells_per_block=(1, 1),
                       feature_vector=True)

print('All baseline descriptors defined.')
test_hog = hog_descriptor(binaries[0])
print(f'HOG dim: {len(test_hog)}')

In [ ]:
# ── Cell 7: AMST v10 Full Descriptor (466 dims) ──────────────────────────
# FIX 3: All comments/prints now consistently say 466 dims.
# C1 APCFW+  : 64 + 16 + 32 + 32 = 144
# C2 Topo MR : 3 resolutions x 30  = 90
# C3 SPD+    : 20x20 upper-tri     = 210
# C5 Compl.  : 22 (actual computed, not 25)
# TOTAL      : 144 + 90 + 210 + 22 = 466

# ── C1: Adaptive Phase-Coherent Fourier-Wavelet Fusion (144-dim) ────────────
def phase_coherent_fourier_wavelet_plus(contour, K_F=64, n_wbands=32):
    r = np.sqrt((contour**2).sum(axis=1))
    F = np.fft.fft(r); mags = np.abs(F); phases = np.angle(F)
    denom = mags[1] if mags[1] > 1e-8 else mags.max() + 1e-12
    mag_n = mags / denom
    fd = mag_n[1:K_F + 1]; ph = phases[1:17]
    n_star = int(np.argmax(mags[1:K_F + 1])) + 1
    rho = n_star / K_F
    wv = 'db4' if rho < 0.10 else ('db2' if rho < 0.25 else 'haar')
    x, yc = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    kc = kappa - kappa.mean()
    max_lv = pywt.dwt_max_level(len(kc), wv)
    L = max(1, min(6, max_lv))
    coeffs = pywt.wavedec(kc, wv, level=L, mode='periodization')
    energies = np.array([np.sum(c**2) for c in coeffs])
    E = energies / (energies.sum() + 1e-12); n_actual = len(E)
    h_idx = np.array([min(l * max(1, n_star // max(n_actual, 1)) + 1, K_F - 1) for l in range(n_actual)])
    cos_ph = np.abs(np.cos(phases[h_idx]))
    Omega = E * cos_ph + 1e-12; Omega /= Omega.sum()
    xi = np.linspace(0, 1, n_actual); xo = np.linspace(0, 1, n_wbands)
    Omega32 = interp1d(xi, Omega, kind='linear')(xo)
    Omega32 = np.maximum(Omega32, 0); Omega32 /= Omega32.sum() + 1e-12
    angles = np.arctan2(contour[:, 0], contour[:, 1])
    ang_hist, _ = np.histogram(angles, bins=32, range=(-np.pi, np.pi))
    ang_dist = ang_hist / (ang_hist.sum() + 1e-12)
    return np.concatenate([fd, ph, Omega32, ang_dist])  # 64+16+32+32=144

# ── C2: Multi-Resolution Topology (90-dim) ───────────────────────────
def multi_resolution_topology(contour, taus=[3, 5, 8]):
    res_points = [128, 256, 512]; all_feats = []
    for res_idx, n_pts in enumerate(res_points):
        if len(contour) > n_pts:
            idx = np.linspace(0, len(contour) - 1, n_pts, dtype=int)
            c_sub = contour[idx]
        else:
            c_sub = contour
        x, yc = c_sub[:, 1], c_sub[:, 0]
        x1 = np.gradient(x); y1 = np.gradient(yc)
        x2 = np.gradient(x1); y2 = np.gradient(y1)
        kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
        kn = (kappa - kappa.min()) / (kappa.max() - kappa.min() + 1e-12)
        N = len(kn)
        tau = min(taus[res_idx], N // 4)
        if N <= tau + 2:
            all_feats.append(np.zeros(30)); continue
        Xk = np.column_stack([kn[:N - tau], kn[tau:]])
        if len(Xk) > 200:
            idxs = np.linspace(0, len(Xk) - 1, 200, dtype=int); Xk = Xk[idxs]
        try:
            dgms = ripser(Xk, maxdim=1)['dgms']
        except:
            all_feats.append(np.zeros(30)); continue
        def vect(dgm, k=7):
            fin = dgm[dgm[:, 1] < np.inf]
            if len(fin) == 0: return np.zeros(k), np.zeros(k), np.zeros(6)
            lt = np.sort(fin[:, 1] - fin[:, 0])[::-1]
            bt = np.sort(fin[:, 0])
            lp = np.zeros(k); lp[:min(len(lt), k)] = lt[:k]
            bp = np.zeros(k); bp[:min(len(bt), k)] = bt[:k]
            tot = lt.sum(); mx = lt[0]; betti = float((lt > 0.01).sum())
            ent = -np.sum(lt / tot * np.log(lt / tot + 1e-12)) if tot > 0 else 0
            med = float(np.median(lt)) if len(lt) > 0 else 0
            return lp, bp, np.array([tot, mx, betti, ent, med, float(len(fin))])
        lt0, bt0, st0 = vect(dgms[0]); lt1, bt1, st1 = vect(dgms[1])
        feat = np.concatenate([lt0, lt1, bt0[:4], bt1[:4], st0, st1[:4]])
        out = np.zeros(30); out[:min(len(feat), 30)] = feat[:30]
        all_feats.append(out)
    return np.concatenate(all_feats)  # 3 x 30 = 90

# ── C3: SPD Manifold Features (210-dim) ───────────────────────────
def spd_manifold_features_v8(contour):
    r = np.sqrt((contour**2).sum(axis=1))
    x, yc = contour[:, 1], contour[:, 0]; N = len(r)
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    t = np.linspace(0, 2 * np.pi, N, endpoint=False)
    rows = [
        r - r.mean(), x - x.mean(), yc - yc.mean(), kappa,
        np.cos(t), np.sin(t),
        ndimage.gaussian_filter1d(r - r.mean(), 2, mode='wrap'),
        ndimage.gaussian_filter1d(r - r.mean(), 8, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 2, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 4, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 8, mode='wrap'),
        np.gradient(kappa),
        ndimage.gaussian_filter1d(r - r.mean(), 4, mode='wrap'),
        ndimage.gaussian_filter1d(kappa, 1, mode='wrap'),
        np.abs(kappa), kappa**2,
        np.sqrt(np.abs(kappa) + 1e-12),
        np.sin(2 * t), np.cos(2 * t),
        np.arctan2(yc, x),
    ]
    d_spd = 20
    fm = np.array(rows[:d_spd], dtype=float)
    fm -= fm.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(fm, axis=1, keepdims=True)
    fm /= (norms + 1e-12)
    S = (fm @ fm.T) / (N - 1) + 1e-4 * np.eye(d_spd)
    ev, evec = np.linalg.eigh(S)
    ev = np.maximum(ev, 1e-8)
    log_S = evec @ np.diag(np.log(ev)) @ evec.T
    return log_S[np.triu_indices(d_spd)]  # 20*21/2 = 210

# ── C5: Shape Complexity Features (22-dim) ───────────────────────────
# FIX 3: Removed the erroneous [:25] slice — now returns exactly 22 dims.
def shape_complexity_features(contour, binary):
    from scipy.spatial import ConvexHull
    from skimage.measure import regionprops
    feats = []
    # Convex hull metrics (3)
    hull = ConvexHull(contour)
    hull_area = hull.volume; hull_perim = hull.area
    contour_area = np.abs(np.sum(contour[:-1, 0] * contour[1:, 1]
                                 - contour[1:, 0] * contour[:-1, 1])) / 2
    contour_perim = np.sum(np.sqrt(np.diff(contour[:, 0])**2 + np.diff(contour[:, 1])**2))
    feats.extend([hull_perim / (contour_perim + 1e-12),           # convexity
                  contour_area / (hull_area + 1e-12),              # solidity
                  4 * np.pi * contour_area / (contour_perim**2 + 1e-12)])  # compactness
    # Region properties (7)
    labeled = measure.label(binary); props = regionprops(labeled)
    if props:
        p = props[0]
        feats.append(float(p.euler_number))
        feats.append(p.major_axis_length / (p.minor_axis_length + 1e-12))
        feats.append(p.extent); feats.append(p.eccentricity)
        feats.append(p.equivalent_diameter_area / max(binary.shape))
        feats.append(p.perimeter / (contour_perim + 1e-12))
        feats.append(p.area / (binary.shape[0] * binary.shape[1]))
    else:
        feats.extend([0.0] * 7)
    # Curvature statistics (7)
    x, yc = contour[:, 1], contour[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    feats.extend([np.mean(kappa), np.std(kappa), np.max(np.abs(kappa)),
                  np.sum(np.diff(np.sign(kappa)) != 0) / len(kappa),
                  float(scipy.stats.skew(kappa)),
                  float(scipy.stats.kurtosis(kappa)),
                  float(np.percentile(np.abs(kappa), 90))])
    # Centroid-distance statistics (5)
    d = np.sqrt((contour**2).sum(axis=1))
    feats.extend([np.std(d), float(np.max(d) - np.min(d)), np.mean(d),
                  float(np.percentile(d, 25)), float(np.percentile(d, 75))])
    # Total: 3+7+7+5 = 22 dims — NO slicing needed
    return np.array(feats, dtype=float)

# ── C4: Multi-Head Fisher Band Attention (fitted per fold — no leakage) ──────
class MultiHeadFisherBandAttention:
    def __init__(self, n_heads=8, n_bands=16, top_k_ratio=0.50):
        self.n_heads = n_heads; self.n_bands = n_bands
        self.top_k_ratio = top_k_ratio
        self.attention_weights = None; self.band_boundaries = None

    def fit(self, X_train, y_train):
        N, D = X_train.shape
        bs = D // self.n_bands
        self.band_boundaries = [(i * bs, min((i + 1) * bs, D)) for i in range(self.n_bands)]
        cls = np.unique(y_train); k = max(1, int(self.n_bands * self.top_k_ratio))
        head_weights = []
        for h in range(self.n_heads):
            np.random.seed(h * 17 + 3)
            sub = np.random.choice(N, int(0.8 * N), replace=False) if N > 20 else np.arange(N)
            Xs, ys = X_train[sub], y_train[sub]
            gms = Xs.mean(axis=0); scores = []
            for b0, b1 in self.band_boundaries:
                Xb = Xs[:, b0:b1]; gmb = gms[b0:b1]; SB = SW = 0.0
                for c in cls:
                    mk = ys == c
                    if mk.sum() < 2: continue
                    mc = Xb[mk].mean(axis=0)
                    SB += mk.sum() * np.dot(mc - gmb, mc - gmb)
                    SW += np.sum((Xb[mk] - mc)**2)
                scores.append(SB / (SW + 1e-8))
            sc = np.array(scores); sc = np.exp(sc - sc.max()); sc /= sc.sum()
            sp = np.zeros(self.n_bands)
            top = np.argsort(sc)[::-1][:k]; sp[top] = sc[top]; sp /= sp.sum() + 1e-12
            head_weights.append(sp)
        avg = np.mean(head_weights, axis=0); avg /= avg.sum() + 1e-12
        self.attention_weights = np.zeros(D)
        for i, (b0, b1) in enumerate(self.band_boundaries):
            self.attention_weights[b0:b1] = avg[i]
        return self

    def transform(self, X):
        if self.attention_weights is None:
            raise ValueError('Call fit() first')
        return X * self.attention_weights[np.newaxis, :]

# ── Full AMST v10 Descriptor (466-dim) ──────────────────────────
def amst_descriptor_v9(contour, binary):
    """Returns 466-dim AMST descriptor: C1(144) + C2(90) + C3(210) + C5(22)."""
    c1 = phase_coherent_fourier_wavelet_plus(contour)  # 144
    c2 = multi_resolution_topology(contour)             # 90
    c3 = spd_manifold_features_v8(contour)              # 210
    c5 = shape_complexity_features(contour, binary)     # 22
    return np.concatenate([c1, c2, c3, c5])             # 466

# ── Dimension verification ──────────────────────────────
test_amst = amst_descriptor_v9(contours[0], binaries[0])
c1_d = len(phase_coherent_fourier_wavelet_plus(contours[0]))
c2_d = len(multi_resolution_topology(contours[0]))
c3_d = len(spd_manifold_features_v8(contours[0]))
c5_d = len(shape_complexity_features(contours[0], binaries[0]))
print(f'  C1 APCFW+ : {c1_d} (expected 144)')
print(f'  C2 Topo MR: {c2_d} (expected 90)')
print(f'  C3 SPD+   : {c3_d} (expected 210)')
print(f'  C5 Compl. : {c5_d} (expected 22)')
print(f'  TOTAL     : {len(test_amst)} (expected {AMST_DIM})')
assert len(test_amst) == AMST_DIM, f'AMST dim wrong: {len(test_amst)} != {AMST_DIM}'
assert c5_d == 22, f'C5 dim wrong: {c5_d} — check shape_complexity_features()'
print(f'AMST v10 descriptor: {AMST_DIM}-dim — verified.')

In [ ]:
# ── Figure 3: APCFW+ Analysis ─────────────────────────────────
display_cls3 = [c for c in [0, 3, 7, 12] if c < n_classes][:4]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Figure 3: Phase-Coherent Fourier-Wavelet Fusion (APCFW+)\n'
             'Top: Fourier Magnitude (64 harmonics) | Bottom: Angular Distribution (32 bins)',
             fontsize=12, fontweight='bold', y=1.01)
K_F = 64
for col, cls in enumerate(display_cls3):
    idx = np.where(y == cls)[0][0]; cnt = contours[idx]
    cls_name = le.classes_[cls].capitalize()
    r = np.sqrt((cnt**2).sum(axis=1))
    F = np.fft.fft(r); mags = np.abs(F)
    denom = mags[1] if mags[1] > 1e-8 else mags.max() + 1e-12
    mag_n = mags / denom
    n_star = int(np.argmax(mags[1:K_F + 1])) + 1
    axes[0, col].bar(range(1, K_F + 1), mag_n[1:K_F + 1], color='#5B7FA6', width=0.8)
    axes[0, col].plot(n_star, mag_n[n_star], 'ro', ms=8, zorder=5, label=f'n*={n_star}')
    axes[0, col].set_title(cls_name, fontsize=9, fontweight='bold')
    axes[0, col].set_xlabel('Harmonic n'); axes[0, col].legend(fontsize=7); axes[0, col].grid(alpha=0.3)
    angles = np.arctan2(cnt[:, 0], cnt[:, 1])
    ang_hist, _ = np.histogram(angles, bins=32, range=(-np.pi, np.pi))
    ang_dist = ang_hist / (ang_hist.sum() + 1e-12)
    axes[1, col].bar(range(32), ang_dist, color='#E84040', width=0.8)
    axes[1, col].set_title('Angular Distribution (32 bins)', fontsize=8)
    axes[1, col].set_xlabel('Angle bin'); axes[1, col].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/fig3_apcfw_analysis.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 3 saved.')

In [ ]:
# ── Figure 4: Topological Persistence Diagrams ─────────────────────────
display_cls4 = [c for c in [0, 3, 7, 12] if c < n_classes][:4]
tau = 5
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
fig.suptitle('Figure 4: Multi-Resolution Topological Persistence\n'
             'Top: Curvature kappa(t) | Bottom: H0/H1 Persistence Diagrams',
             fontsize=12, fontweight='bold', y=1.01)
for col, cls in enumerate(display_cls4):
    idx = np.where(y == cls)[0][0]; cnt = contours[idx]
    cls_name = le.classes_[cls].capitalize()
    x, yc = cnt[:, 1], cnt[:, 0]
    x1 = np.gradient(x); y1 = np.gradient(yc)
    x2 = np.gradient(x1); y2 = np.gradient(y1)
    kappa = (x1 * y2 - x2 * y1) / (x1**2 + y1**2 + 1e-12)**1.5
    kappa_n = (kappa - kappa.min()) / (kappa.max() - kappa.min() + 1e-12)
    N = len(kappa_n); t_ax = np.linspace(0, 1, N)
    axes[0, col].plot(t_ax, kappa, 'g-', lw=1.5)
    axes[0, col].fill_between(t_ax, kappa, alpha=0.2, color='green')
    axes[0, col].axhline(0, color='k', lw=0.5, ls='--')
    axes[0, col].set_title(cls_name, fontsize=10, fontweight='bold')
    axes[0, col].set_xlabel('t'); axes[0, col].grid(alpha=0.3)
    Xk = np.column_stack([kappa_n[:N - tau], kappa_n[tau:]])
    if len(Xk) > 200:
        idxs = np.linspace(0, len(Xk) - 1, 200, dtype=int); Xk = Xk[idxs]
    dgms = ripser(Xk, maxdim=1)['dgms']
    ax = axes[1, col]
    h0 = dgms[0]; h0f = h0[h0[:, 1] < np.inf]
    if len(h0f): ax.scatter(h0f[:, 0], h0f[:, 1], c='#5B7FA6', s=40, label='H0', alpha=0.8)
    h1 = dgms[1]; h1f = h1[h1[:, 1] < np.inf]
    if len(h1f): ax.scatter(h1f[:, 0], h1f[:, 1], c='#E84040', marker='^', s=60, label='H1', alpha=0.8)
    lim = max(0.5, ax.get_xlim()[1] if ax.get_xlim()[1] > 0 else 0.5)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel('Birth'); ax.set_ylabel('Death')
    ax.set_title(f'Persistence H0:{len(h0f)} H1:{len(h1f)}', fontsize=8)
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/fig4_persistence_diagrams.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 4 saved.')

In [ ]:
# ── Cell 8: Feature Extraction (All 8 Methods) ──────────────────────────
print(f'Extracting features (N={len(contours)} images, all methods)...')
N = len(contours)
fd_f=[]; wd_f=[]; hy_f=[]; ze_f=[]; cs_f=[]; sc_f=[]; hg_f=[]; am_f=[]
for i in tqdm(range(N), desc='Features'):
    cnt = contours[i]; bimg = binaries[i]
    try:    fd_f.append(fourier_descriptor(cnt))
    except: fd_f.append(np.zeros(79))
    try:    wd_f.append(wavelet_descriptor(cnt))
    except: wd_f.append(np.zeros(18))
    try:    hy_f.append(simple_hybrid_descriptor(cnt))
    except: hy_f.append(np.zeros(97))
    try:    ze_f.append(zernike_moments(bimg))
    except: ze_f.append(np.zeros(36))
    try:    cs_f.append(curvature_scale_space(cnt))
    except: cs_f.append(np.zeros(20))
    try:    sc_f.append(shape_context(cnt))
    except: sc_f.append(np.zeros(60))
    try:    hg_f.append(hog_descriptor(bimg))
    except: hg_f.append(np.zeros(576))
    try:    am_f.append(amst_descriptor_v9(cnt, bimg))
    except: am_f.append(np.zeros(AMST_DIM))

X_fd      = np.array(fd_f)
X_wd      = np.array(wd_f)
X_hybrid  = np.array(hy_f)
X_zernike = np.array(ze_f)
X_css     = np.array(cs_f)
X_sc      = np.array(sc_f)
X_hog     = np.array(hg_f)
X_amst    = np.nan_to_num(np.array(am_f))

print('\nFeature dimensions:')
for nm, X in [('Fourier', X_fd), ('Wavelet', X_wd), ('F+W Hybrid', X_hybrid),
              ('Zernike', X_zernike), ('CSS', X_css), ('Shape Context', X_sc),
              ('HOG', X_hog), (f'AMST v10 ({AMST_DIM}d)', X_amst)]:
    print(f'  {nm:25s}: {X.shape}')
assert X_amst.shape[1] == AMST_DIM, f'AMST dim check failed: {X_amst.shape[1]} != {AMST_DIM}'
print(f'All dimension assertions passed.')

In [ ]:
# ── Cell 9: 5-Fold CV Evaluation ─────────────────────────────────
# FIX 1: ALL methods use the same SVM-RBF classifier for fair comparison.
#         Stacking ensemble is run SEPARATELY for AMST and reported as an
#         additional experiment, not mixed into the main comparison table.

def find_best_svm(X_tr, y_tr):
    param_grid = {'C': [0.1, 1, 10, 100, 500, 1000],
                  'gamma': ['scale', 'auto', 0.001, 0.01, 0.1]}
    gs = GridSearchCV(SVC(kernel='rbf', decision_function_shape='ovr'),
                      param_grid, cv=3, scoring='accuracy', n_jobs=-1)
    gs.fit(X_tr, y_tr)
    return gs.best_params_

def evaluate_descriptor_svm(X, y, name, n_splits=5, use_attention=False):
    """Universal SVM-RBF evaluator used for ALL methods (fair comparison)."""
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; precs=[]; recs=[]; fold_accs=[]
    for tr, te in skf.split(X, y):
        X_tr, X_te = X[tr], X[te]; y_tr, y_te = y[tr], y[te]
        if use_attention:
            attn = MultiHeadFisherBandAttention(8, 16, 0.50)
            attn.fit(X_tr, y_tr)
            X_tr = attn.transform(X_tr); X_te = attn.transform(X_te)
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
        bp = find_best_svm(X_tr_s, y_tr)
        clf = SVC(kernel='rbf', decision_function_shape='ovr', **bp)
        clf.fit(X_tr_s, y_tr); yp = clf.predict(X_te_s)
        accs.append(accuracy_score(y_te, yp))
        f1s.append(f1_score(y_te, yp, average='macro', zero_division=0))
        precs.append(precision_score(y_te, yp, average='macro', zero_division=0))
        recs.append(recall_score(y_te, yp, average='macro', zero_division=0))
        fold_accs.append(accuracy_score(y_te, yp))
    return {'Method': name, 'Accuracy': np.mean(accs), 'Accuracy_std': np.std(accs),
            'F1_macro': np.mean(f1s), 'Precision_macro': np.mean(precs),
            'Recall_macro': np.mean(recs), 'Dim': X.shape[1], 'fold_accs': fold_accs}

def stacking_ensemble_predict(X_train, y_train, X_test):
    """Stacking ensemble: RF + XGBoost base → LogisticRegression meta."""
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
    xgb_clf = (xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
               random_state=SEED, n_jobs=-1, verbosity=0) if XGB_AVAILABLE else None)
    rf_preds = cross_val_predict(rf, X_train, y_train, cv=5, method='predict_proba')
    if xgb_clf is not None:
        xgb_preds = cross_val_predict(xgb_clf, X_train, y_train, cv=5, method='predict_proba')
        meta_train = np.column_stack([rf_preds, xgb_preds])
    else:
        meta_train = rf_preds
    rf.fit(X_train, y_train); rf_test = rf.predict_proba(X_test)
    if xgb_clf is not None:
        xgb_clf.fit(X_train, y_train)
        meta_test = np.column_stack([rf_test, xgb_clf.predict_proba(X_test)])
    else:
        meta_test = rf_test
    meta = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=SEED)
    meta.fit(meta_train, y_train)
    return meta.predict(meta_test)

def evaluate_descriptor_stacking(X, y, name, n_splits=5, use_attention=False):
    """Stacking ensemble evaluator — SEPARATE from the fair SVM comparison."""
    X = np.nan_to_num(X.copy())
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    accs=[]; f1s=[]; precs=[]; recs=[]; fold_accs=[]
    for tr, te in skf.split(X, y):
        X_tr, X_te = X[tr], X[te]; y_tr, y_te = y[tr], y[te]
        if use_attention:
            attn = MultiHeadFisherBandAttention(8, 16, 0.50)
            attn.fit(X_tr, y_tr)
            X_tr = attn.transform(X_tr); X_te = attn.transform(X_te)
        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)
        yp = stacking_ensemble_predict(X_tr_s, y_tr, X_te_s)
        accs.append(accuracy_score(y_te, yp))
        f1s.append(f1_score(y_te, yp, average='macro', zero_division=0))
        precs.append(precision_score(y_te, yp, average='macro', zero_division=0))
        recs.append(recall_score(y_te, yp, average='macro', zero_division=0))
        fold_accs.append(accuracy_score(y_te, yp))
    return {'Method': name, 'Accuracy': np.mean(accs), 'Accuracy_std': np.std(accs),
            'F1_macro': np.mean(f1s), 'Precision_macro': np.mean(precs),
            'Recall_macro': np.mean(recs), 'Dim': X.shape[1], 'fold_accs': fold_accs}

# ── FIX 1: All methods use SVM → fair comparison table ─────────────────
print('Running fair 5-fold CV (ALL methods SVM-RBF, attention per-fold)...\n')
svm_results = []
method_list = [
    (X_fd,      'Fourier Descriptor',  False),
    (X_wd,      'Wavelet Descriptor',  False),
    (X_hybrid,  'Simple F+W Hybrid',   False),
    (X_zernike, 'Zernike Moments',     False),
    (X_css,     'CSS Descriptor',      False),
    (X_sc,      'Shape Context',       False),
    (X_hog,     'HOG + SVM',           False),
    (X_amst,    'AMST (Proposed)',      True),
]
for X_feat, nm, use_attn in method_list:
    res = evaluate_descriptor_svm(X_feat, y, nm, use_attention=use_attn)
    svm_results.append(res)
    print(f'{nm:28s} | Acc: {res["Accuracy"]:.4f}±{res["Accuracy_std"]:.4f} | F1: {res["F1_macro"]:.4f}')

svm_df = pd.DataFrame(svm_results)
amst_acc_svm  = svm_df[svm_df['Method'] == 'AMST (Proposed)']['Accuracy'].values[0] * 100
best_base_svm = svm_df[svm_df['Method'] != 'AMST (Proposed)']['Accuracy'].max() * 100

print(f'\n=== FAIR Comparison (SVM-RBF, same classifier for all) ===')
print(f'AMST (SVM)   : {amst_acc_svm:.2f}%')
print(f'Best Baseline: {best_base_svm:.2f}%')
print(f'AMST Gain    : +{amst_acc_svm - best_base_svm:.2f} pp')

# ── Additional: Stacking Ensemble for AMST only ───────────────────
print('\nRunning AMST stacking ensemble (additional experiment)...')
stack_result = evaluate_descriptor_stacking(X_amst, y, 'AMST (Stacking)', use_attention=True)
amst_acc_stack = stack_result['Accuracy'] * 100
print(f'AMST (Stacking): {amst_acc_stack:.2f}% | Stacking gain over SVM: +{amst_acc_stack - amst_acc_svm:.2f} pp')

# ── Combined results DataFrame ────────────────────────────
all_results_df = pd.DataFrame(svm_results + [stack_result])
print('\nAll results:')
print(svm_df[['Method','Accuracy','Accuracy_std','F1_macro','Dim']].to_string(index=False))

In [ ]:
# ── Cell 10: Statistical Significance (Paired t-test, AMST SVM vs Baselines) ──
amst_folds = np.array(svm_df[svm_df['Method'] == 'AMST (Proposed)']['fold_accs'].values[0])

print('Paired t-test: AMST (SVM) vs each baseline (5 fold-level accuracies)\n')
print(f'{"Method":<28} {"AMST%":>9} {"Base%":>9} {"Delta(pp)":>10} {"t-stat":>8} {"p-value":>10} {"Sig?":>6}')
print('-' * 80)
stat_rows = []
for _, row in svm_df.iterrows():
    nm = row['Method']
    if nm == 'AMST (Proposed)': continue
    bf = np.array(row['fold_accs'])
    t_stat, p_val = scipy.stats.ttest_rel(amst_folds, bf)
    delta = (amst_folds.mean() - bf.mean()) * 100
    sig = '*' if p_val < 0.05 else ' '
    print(f'{nm:<28} {amst_folds.mean()*100:>8.2f}% {bf.mean()*100:>8.2f}% {delta:>+9.2f}pp {t_stat:>8.3f} {p_val:>10.4f} {sig:>6}')
    stat_rows.append({'Baseline': nm, 'AMST_%': amst_folds.mean()*100,
                      'Base_%': bf.mean()*100, 'Delta_pp': delta,
                      't_stat': t_stat, 'p_value': p_val, 'Significant': p_val < 0.05})

stat_df = pd.DataFrame(stat_rows)
n_sig = stat_df['Significant'].sum()
print(f'\nAMST fold accuracies: {[f"{a:.3f}" for a in amst_folds]}')
print(f'AMST outperforms {n_sig}/{len(stat_df)} baselines significantly (p<0.05, paired t-test)')

In [ ]:
# ── Figure 5: Classification Performance Comparison ────────────────
# FIX 1: All bars use SVM — a genuinely fair comparison.
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Figure 5: Classification Performance — KIMIA-216 | 5-fold CV\n'
             'FIX: ALL methods use same SVM-RBF classifier (fair comparison). '
             'Stacking result shown separately in panel C.',
             fontsize=12, fontweight='bold')

methods = svm_df['Method'].values
accs    = svm_df['Accuracy'].values * 100
stds    = svm_df['Accuracy_std'].values * 100
f1s     = svm_df['F1_macro'].values * 100
colors  = ['#5B7FA6'] * (len(methods) - 1) + ['#E84040']

# (A) Accuracy
bars = axes[0].barh(methods, accs, xerr=stds, color=colors, edgecolor='white', capsize=4, height=0.65)
axes[0].set_xlabel('Accuracy (%)', fontsize=11); axes[0].set_xlim(0, 115)
axes[0].set_title('(A) Accuracy ± SD\n[All methods: SVM-RBF — fair]', fontsize=10)
axes[0].grid(axis='x', alpha=0.3)
for i, (bar, acc, std) in enumerate(zip(bars, accs, stds)):
    fw = 'bold' if i == len(methods) - 1 else 'normal'
    axes[0].text(acc + std + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{acc:.1f}%', va='center', fontsize=9, fontweight=fw)

# (B) F1-Score
bars2 = axes[1].barh(methods, f1s, color=colors, edgecolor='white', height=0.65)
axes[1].set_xlabel('Macro F1 (%)', fontsize=11); axes[1].set_xlim(0, 115)
axes[1].set_title('(B) Macro F1\n[All methods: SVM-RBF — fair]', fontsize=10)
axes[1].grid(axis='x', alpha=0.3)
for i, (bar, f1) in enumerate(zip(bars2, f1s)):
    fw = 'bold' if i == len(methods) - 1 else 'normal'
    axes[1].text(f1 + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{f1:.1f}%', va='center', fontsize=9, fontweight=fw)

# (C) AMST: SVM vs Stacking (separate, clearly labelled)
amst_variants = ['AMST (SVM) — descriptor only gain',
                 'AMST (Stacking) — descriptor + classifier gain']
amst_accs = [amst_acc_svm, amst_acc_stack]
colors_c = ['#C0392B', '#E84040']
bars3 = axes[2].barh(amst_variants, amst_accs, color=colors_c, edgecolor='white', height=0.45)
for bar, acc in zip(bars3, amst_accs):
    axes[2].text(acc + 0.5, bar.get_y() + bar.get_height() / 2,
                 f'{acc:.1f}%', va='center', fontsize=10, fontweight='bold')
axes[2].axvline(best_base_svm, color='navy', ls='--', lw=1.5,
                label=f'Best Baseline = {best_base_svm:.1f}%')
axes[2].set_title('(C) AMST SVM vs Stacking\n[Stacking is an additional bonus, not in main table]',
                  fontsize=10)
axes[2].set_xlim(0, 115); axes[2].grid(axis='x', alpha=0.3); axes[2].legend(fontsize=9)

fig.legend(handles=[mpatches.Patch(color='#5B7FA6', label='Baseline (SVM)'),
                    mpatches.Patch(color='#E84040', label='AMST (Proposed)')],
           loc='lower center', ncol=2, fontsize=10)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('/content/fig5_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 5 saved.')

In [ ]:
# ── Figure 6: Confusion Matrix (5-fold aggregated, AMST SVM) ─────────
skf_cm = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_ac = np.nan_to_num(X_amst.copy()); all_yt=[]; all_yp=[]
for tr, te in skf_cm.split(X_ac, y):
    X_tr, X_te = X_ac[tr], X_ac[te]; y_tr, y_te = y[tr], y[te]
    attn_cm = MultiHeadFisherBandAttention(8, 16, 0.50)
    attn_cm.fit(X_tr, y_tr)
    X_tr = attn_cm.transform(X_tr); X_te = attn_cm.transform(X_te)
    sc_cm = RobustScaler()
    X_tr_s = sc_cm.fit_transform(X_tr); X_te_s = sc_cm.transform(X_te)
    bp_cm = find_best_svm(X_tr_s, y_tr)
    clf_cm = SVC(kernel='rbf', decision_function_shape='ovr', **bp_cm)
    clf_cm.fit(X_tr_s, y_tr)
    yp_cm = clf_cm.predict(X_te_s)
    all_yt.extend(y_te.tolist()); all_yp.extend(yp_cm.tolist())

all_yt = np.array(all_yt); all_yp = np.array(all_yp)
cm_acc = accuracy_score(all_yt, all_yp)
cm = confusion_matrix(all_yt, all_yp)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
print(f'5-fold aggregated accuracy (SVM): {cm_acc*100:.2f}%')

fig, ax = plt.subplots(figsize=(13, 11))
im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='Recognition Rate (%)')
cn = [le.classes_[i].capitalize() for i in range(n_classes)]
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels(cn, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(cn, fontsize=9)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, f'{cm_pct[i,j]:.0f}', ha='center', va='center', fontsize=8,
                color='white' if cm_pct[i, j] > 60 else 'black',
                fontweight='bold' if i == j else 'normal')
ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Figure 6: AMST v10 Confusion Matrix (5-fold CV aggregated)\n'
             f'SVM-RBF + Multi-Head Attention | Acc: {cm_acc*100:.2f}%',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/fig6_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 6 saved.')

In [ ]:
# ── Cell 11: Ablation Study ─────────────────────────────────
# FIX 4: ALL ablation sub-variants use SVM so each step adds exactly one
#         descriptor component. Stacking gain shown as a final separate row.

def extract_ablation_variant(contour, binary, variant):
    c1 = phase_coherent_fourier_wavelet_plus(contour)
    c2 = multi_resolution_topology(contour)
    c3 = spd_manifold_features_v8(contour)
    c5 = shape_complexity_features(contour, binary)
    if variant == 'c1_only':    return c1
    elif variant == 'c1_c2':    return np.concatenate([c1, c2])
    elif variant == 'c1_c2_c3': return np.concatenate([c1, c2, c3])
    else:                        return np.concatenate([c1, c2, c3, c5])

print('Extracting ablation variants...')
variant_map = [
    ('c1_only',   'C1: APCFW+ only (144d)',          144),
    ('c1_c2',     'C1+C2: +Topology MR (234d)',       234),
    ('c1_c2_c3',  'C1+C2+C3: +SPD Manifold (444d)',  444),
    ('full_amst', 'Full AMST C1+C2+C3+C5 (466d)',    466),
]
abl_feats = {}
for vname, vlab, expected_dim in variant_map:
    feats = []
    for i in tqdm(range(len(contours)), desc=vlab, leave=False):
        try:    feats.append(extract_ablation_variant(contours[i], binaries[i], vname))
        except: feats.append(np.zeros(expected_dim))
    abl_feats[vlab] = np.nan_to_num(np.array(feats))

# SVM for every sub-variant (FIX 4)
print('\nAblation 5-fold CV (SVM-RBF for all — fair incremental analysis):')
abl_svm_results = []
for vlab, Xa in abl_feats.items():
    res = evaluate_descriptor_svm(Xa, y, vlab, use_attention=True)
    abl_svm_results.append(res)
    print(f'  {vlab:<45} Acc: {res["Accuracy"]*100:.2f}±{res["Accuracy_std"]*100:.2f}%')

# Stacking result for Full AMST as a final bonus row
abl_stack_res = evaluate_descriptor_stacking(abl_feats['Full AMST C1+C2+C3+C5 (466d)'],
                                              y, 'Full AMST + Stacking (bonus)', use_attention=True)
abl_svm_results.append(abl_stack_res)
print(f'  {"Full AMST + Stacking (bonus)":<45} Acc: {abl_stack_res["Accuracy"]*100:.2f}±{abl_stack_res["Accuracy_std"]*100:.2f}%  [Stacking bonus, not in main table]')

abl_df = pd.DataFrame(abl_svm_results)

# Incremental significance
print('\nIncremental significance (paired t-test):')
abl_folds = [np.array(r['fold_accs']) for r in abl_svm_results]
abl_names = [r['Method'] for r in abl_svm_results]
for i in range(1, len(abl_folds)):
    t_stat, p_val = scipy.stats.ttest_rel(abl_folds[i], abl_folds[i - 1])
    delta = (abl_folds[i].mean() - abl_folds[i - 1].mean()) * 100
    sig = '*' if p_val < 0.05 else ' '
    print(f'  {abl_names[i-1][:30]:<30} -> {abl_names[i][:30]:<30} {delta:+>7.2f}pp  p={p_val:.4f} {sig}')

In [ ]:
# ── Figure 7: Ablation Study ─────────────────────────────────
abl_names_plot = abl_df['Method'].values
abl_accs  = abl_df['Accuracy'].values * 100
abl_stds  = abl_df['Accuracy_std'].values * 100
colors_abl = ['#AED6F1', '#5DADE2', '#2471A3', '#1A5276', '#E84040'][:len(abl_names_plot)]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(abl_names_plot, abl_accs, xerr=abl_stds, color=colors_abl,
               edgecolor='white', capsize=4, height=0.6)
ax.axvline(abl_accs[-2], color='#1A5276', ls='--', lw=1.5, alpha=0.7,
           label=f'Full AMST SVM = {abl_accs[-2]:.1f}%')
ax.axvline(abl_accs[-1], color='#E84040', ls=':', lw=1.5, alpha=0.7,
           label=f'+ Stacking (bonus) = {abl_accs[-1]:.1f}%')
for i, (bar, acc, std) in enumerate(zip(bars, abl_accs, abl_stds)):
    fw = 'bold' if i >= len(bars) - 2 else 'normal'
    ax.text(acc + std + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{acc:.2f}%', va='center', fontsize=10, fontweight=fw)
ax.set_xlabel('Accuracy (%)', fontsize=12)
ax.set_title('Figure 7: Ablation Study — Incremental Component Gains (v10)\n'
             'FIX 4: All sub-variants use SVM-RBF; stacking is a final separate row.',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 115); ax.grid(axis='x', alpha=0.3); ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('/content/fig7_ablation_study.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 7 saved.')

In [ ]:
# ── Cell 12: Robustness — Gaussian Contour Noise ───────────────────
noise_levels = [0.0, 0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25]
noise_methods = [
    (X_fd,     'Fourier Descriptor'),
    (X_wd,     'Wavelet Descriptor'),
    (X_hybrid, 'Simple F+W Hybrid'),
    (X_css,    'CSS Descriptor'),
    (X_sc,     'Shape Context'),
    (X_amst,   'AMST (Proposed)'),
]
styles = [('--','o','#5B7FA6'), ('--','s','#E8A020'), ('--','^','#27AE60'),
          ('--','D','#8E44AD'), ('--','v','#34495E'), ('-','*','#E84040')]

def noisy_cnt(cnt, sigma):
    return center_and_scale(cnt + np.random.normal(0, sigma, cnt.shape))

def eval_noisy(X_orig, y, sigma, nm, n_runs=3):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 7 + 13)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            cn = noisy_cnt(contours[i], sigma)
            try:
                if 'AMST' in nm:    feat = amst_descriptor_v9(cn, binaries[i])
                elif 'Fourier' in nm: feat = fourier_descriptor(cn)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(cn)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(cn)
                elif 'CSS' in nm:   feat = curvature_scale_space(cn)
                else:               feat = shape_context(cn)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_n = RobustScaler()
        clf_n = SVC(kernel='rbf', C=100, gamma='scale')
        clf_n.fit(sc_n.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_n.predict(sc_n.transform(Xte))))
    return np.mean(accs)

print('Computing noise robustness...')
noise_res = {nm: [] for _, nm in noise_methods}
for sigma in tqdm(noise_levels, desc='Noise levels'):
    for Xf, nm in noise_methods:
        noise_res[nm].append(eval_noisy(Xf, y, sigma, nm))
print('Done.')

In [ ]:
# ── Figure 8: Noise Robustness ─────────────────────────────
# FIX 2: Y-axis now correctly shows 0–100% (multiply raw accuracy by 100).
fig, ax = plt.subplots(figsize=(11, 6))
for (_, nm), (ls, mk, col) in zip(noise_methods, styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot(noise_levels, [v * 100 for v in noise_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Gaussian Noise σ', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Figure 8: Robustness to Gaussian Contour Noise', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('/content/fig8_noise_robustness.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 8 saved.')

In [ ]:
# ── Cell 13: Robustness — Partial Occlusion ───────────────────
occ_levels = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]

def occ_binary_fn(binary, frac):
    if frac == 0: return binary
    h, w = binary.shape; side = np.sqrt(frac)
    ch, cw = int(h * side), int(w * side)
    r0 = max(0, (h - ch) // 2); c0 = max(0, (w - cw) // 2)
    occ = binary.copy(); occ[r0:r0 + ch, c0:c0 + cw] = 0
    return occ

def eval_occ(X_orig, y, frac, nm, n_runs=3):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 11 + 7)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            ob = occ_binary_fn(binaries[i], frac)
            try:
                oc = center_and_scale(extract_contour(ob))
                if 'AMST' in nm:    feat = amst_descriptor_v9(oc, ob)
                elif 'Fourier' in nm: feat = fourier_descriptor(oc)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(oc)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(oc)
                elif 'CSS' in nm:   feat = curvature_scale_space(oc)
                else:               feat = shape_context(oc)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_o = RobustScaler()
        clf_o = SVC(kernel='rbf', C=100, gamma='scale')
        clf_o.fit(sc_o.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_o.predict(sc_o.transform(Xte))))
    return np.mean(accs)

print('Computing occlusion robustness...')
occ_res = {nm: [] for _, nm in noise_methods}
for frac in tqdm(occ_levels, desc='Occlusion levels'):
    for Xf, nm in noise_methods:
        occ_res[nm].append(eval_occ(Xf, y, frac, nm))
print('Done.')

In [ ]:
# ── Figure 9: Occlusion Robustness ─────────────────────────
# FIX 2: Y-axis now correctly shows 0–100%.
fig, ax = plt.subplots(figsize=(11, 6))
for (_, nm), (ls, mk, col) in zip(noise_methods, styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot([f * 100 for f in occ_levels], [v * 100 for v in occ_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Occlusion Level (% image area)', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Figure 9: Robustness to Partial Occlusion', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('/content/fig9_occlusion_robustness.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 9 saved.')

In [ ]:
# ── Cell 14: Shape Retrieval (MAP) ──────────────────────
def retrieval_pr(X, y):
    Xs = StandardScaler().fit_transform(np.nan_to_num(X))
    N = len(y); APs=[]; allP=[]; allR=[]
    for qi in range(N):
        dists = np.sqrt(((Xs - Xs[qi])**2).sum(axis=1))
        ranked = np.argsort(dists)[1:]
        rel = (y[ranked] == y[qi]).astype(int)
        if rel.sum() == 0: continue
        cs = np.cumsum(rel); pos = np.arange(1, len(ranked) + 1)
        APs.append((cs / pos * rel).sum() / rel.sum())
        allP.append(cs / pos); allR.append(cs / rel.sum())
    rg = np.linspace(0, 1, 20)
    ip = [np.interp(rg, r, p) for p, r in zip(allP, allR)]
    return rg, np.mean(ip, axis=0), np.mean(APs)

print('Computing retrieval PR curves (full leave-one-out)...')
ret_methods = [
    (X_fd,      'Fourier Descriptor'),
    (X_wd,      'Wavelet Descriptor'),
    (X_hybrid,  'Simple F+W Hybrid'),
    (X_zernike, 'Zernike Moments'),
    (X_sc,      'Shape Context'),
    (X_amst,    'AMST (Proposed)'),
]
pr_curves = {}
for Xf, nm in tqdm(ret_methods, desc='Retrieval'):
    rg, mp, MAP = retrieval_pr(Xf, y)
    pr_curves[nm] = (rg, mp, MAP)
    print(f'  {nm:28s} MAP = {MAP:.4f}')

amst_map = pr_curves['AMST (Proposed)'][2]
best_map = max(v[2] for v in pr_curves.values())
print(f'\nAMST MAP: {amst_map:.4f} | Best MAP overall: {best_map:.4f} | AMST is best: {amst_map >= best_map}')

In [ ]:
# ── Figure 10: Precision-Recall Curves ─────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
cpr = ['#5B7FA6', '#E8A020', '#27AE60', '#8E44AD', '#34495E', '#E84040']
lpr = [1.5] * 5 + [2.8]
for (nm, (rc, pr, MAP)), col, lw in zip(pr_curves.items(), cpr, lpr):
    ls = '-' if 'AMST' in nm else '--'
    ax.plot(rc, pr, color=col, lw=lw, label=f'{nm} (MAP={MAP:.3f})', ls=ls)
ax.set_xlabel('Recall', fontsize=12); ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Figure 10: Shape Retrieval Precision-Recall Curves\n'
             '(Leave-one-out, all queries)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('/content/fig10_precision_recall.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 10 saved.')

In [ ]:
# ── Figure 11: Multi-Head Fisher Band Attention (C4) ─────────────
attn_vis = MultiHeadFisherBandAttention(8, 16, 0.50)
attn_vis.fit(np.nan_to_num(X_amst), y)
attn_w = attn_vis.attention_weights; D = len(attn_w)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Figure 11: Multi-Head Fisher-Criterion Band Weighting (C4)\n'
             '[Visualisation uses full-data fit — classification uses per-fold fit]',
             fontsize=11, fontweight='bold')

axes[0].bar(range(D), attn_w, color=plt.cm.viridis(attn_w / (attn_w.max() + 1e-12)), width=1.)
for b0, _ in attn_vis.band_boundaries:
    axes[0].axvline(b0, color='red', ls=':', lw=0.8, alpha=0.6)
axes[0].axvspan(0, 144, alpha=0.07, color='blue', label=f'C1 APCFW+ (144)')
axes[0].axvspan(144, 234, alpha=0.07, color='green', label='C2 Topo MR (90)')
axes[0].axvspan(234, 444, alpha=0.07, color='orange', label='C3 SPD+ (210)')
axes[0].axvspan(444, 466, alpha=0.07, color='purple', label='C5 Complex (22)')
axes[0].set_title('(A) Per-Dim Fisher Weights'); axes[0].legend(fontsize=8)
axes[0].set_xlabel(f'Descriptor Dimension (total {AMST_DIM})')

bw = [attn_w[b0:b1].mean() for b0, b1 in attn_vis.band_boundaries]
axes[1].bar(range(len(bw)), bw, color=['#E84040' if w == max(bw) else '#5B7FA6' for w in bw])
axes[1].set_title('(B) Per-Band Mean Weight'); axes[1].grid(alpha=0.3)

gm = np.nan_to_num(X_amst).mean(axis=0); fs = []
for b0, b1 in attn_vis.band_boundaries:
    Xb = np.nan_to_num(X_amst)[:, b0:b1]; gmb = gm[b0:b1]; SB = SW = 0.
    for c in np.unique(y):
        mk = y == c
        if mk.sum() < 2: continue
        mc = Xb[mk].mean(axis=0)
        SB += mk.sum() * np.dot(mc - gmb, mc - gmb)
        SW += np.sum((Xb[mk] - mc)**2)
    fs.append(SB / (SW + 1e-8))
fn_arr = np.array(fs); fn_arr /= (fn_arr.max() if fn_arr.max() > 0 else 1.)
axes[2].barh(range(len(fn_arr)), fn_arr, color=['#E84040' if v == fn_arr.max() else '#5B7FA6' for v in fn_arr])
axes[2].set_title('(C) Normalised Fisher Criterion per Band'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/fig11_fisher_attention.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 11 saved.')

In [ ]:
# ── Cell 15: Robustness — Rotation ────────────────────────
def rotate_contour(cnt, angle_deg):
    angle = np.radians(angle_deg)
    R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    return center_and_scale(cnt @ R.T)

def eval_rotation(X_orig, y, angle, nm, n_runs=3):
    accs = []; D = X_orig.shape[1]
    for run in range(n_runs):
        np.random.seed(run * 13 + 5)
        itr, ite = np.split(np.random.permutation(len(y)), [int(0.7 * len(y))])
        Xte = []
        for i in ite:
            rc = rotate_contour(contours[i], angle)
            try:
                if 'AMST' in nm:    feat = amst_descriptor_v9(rc, binaries[i])
                elif 'Fourier' in nm: feat = fourier_descriptor(rc)
                elif 'Wavelet' in nm: feat = wavelet_descriptor(rc)
                elif 'Hybrid' in nm: feat = simple_hybrid_descriptor(rc)
                elif 'CSS' in nm:   feat = curvature_scale_space(rc)
                else:               feat = shape_context(rc)
                if len(feat) < D: feat = np.pad(feat, (0, D - len(feat)))
                elif len(feat) > D: feat = feat[:D]
            except: feat = np.zeros(D)
            Xte.append(feat)
        Xte = np.nan_to_num(np.array(Xte))
        sc_r = RobustScaler()
        clf_r = SVC(kernel='rbf', C=100, gamma='scale')
        clf_r.fit(sc_r.fit_transform(np.nan_to_num(X_orig[itr])), y[itr])
        accs.append(accuracy_score(y[ite], clf_r.predict(sc_r.transform(Xte))))
    return np.mean(accs)

rot_methods = [
    (X_fd,     'Fourier Descriptor'),
    (X_wd,     'Wavelet Descriptor'),
    (X_hybrid, 'Simple F+W Hybrid'),
    (X_sc,     'Shape Context'),
    (X_amst,   'AMST (Proposed)'),
]
rot_styles = [('--','o','#5B7FA6'), ('--','s','#E8A020'), ('--','^','#27AE60'),
              ('--','v','#34495E'), ('-','*','#E84040')]
angles = [0, 15, 30, 45, 60, 90, 120, 180]
print('Computing rotation robustness...')
rot_res = {nm: [] for _, nm in rot_methods}
for angle in tqdm(angles, desc='Rotation'):
    for Xf, nm in rot_methods:
        rot_res[nm].append(eval_rotation(Xf, y, angle, nm))
print('Done.')

In [ ]:
# ── Figure 13: Rotation Robustness ───────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
for (_, nm), (ls, mk, col) in zip(rot_methods, rot_styles):
    lw = 2.8 if 'AMST' in nm else 1.5; ms = 10 if 'AMST' in nm else 7
    ax.plot(angles, [v * 100 for v in rot_res[nm]],
            ls=ls, marker=mk, color=col, lw=lw, ms=ms, label=nm)
ax.set_xlabel('Rotation Angle (degrees)', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Figure 13: Robustness to Rotation', fontsize=12, fontweight='bold')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('/content/fig13_rotation.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 13 saved.')

In [ ]:
# ── Figure 12: Comprehensive Performance Dashboard ─────────────
fig = plt.figure(figsize=(22, 18))
fig.suptitle(f'Figure 12: AMST v10 Comprehensive Performance Dashboard — KIMIA-216\n'
             f'({AMST_DIM}-dim | Fair SVM comparison | No data leakage | 5-fold CV)',
             fontsize=15, fontweight='bold', y=1.01)

# (A) 4-Metric comparison
ax1 = fig.add_subplot(3, 3, 1)
metrics = ['Accuracy', 'F1_macro', 'Precision_macro', 'Recall_macro']
amst_row = svm_df[svm_df['Method'] == 'AMST (Proposed)'].iloc[0]
va = [amst_row[m] * 100 for m in metrics]
vb = [svm_df[svm_df['Method'] != 'AMST (Proposed)'][m].max() * 100 for m in metrics]
x = np.arange(len(metrics)); w = 0.35
ax1.bar(x - w/2, vb, w, label='Best Baseline (SVM)', color='#5B7FA6', alpha=0.8)
ax1.bar(x + w/2, va, w, label='AMST (SVM)', color='#E84040', alpha=0.9)
ax1.set_xticks(x); ax1.set_xticklabels(['Acc', 'F1', 'Prec', 'Rec'], fontsize=9)
ax1.set_title('(A) 4-Metric Comparison\n[Fair: same SVM classifier]', fontsize=9)
ax1.legend(fontsize=7); ax1.set_ylim(0, 115)
for xi, vi in zip(x, va):
    ax1.text(xi + w/2, vi + 1, f'{vi:.1f}%', ha='center', fontsize=8, color='#C0392B', fontweight='bold')

# (B) MAP retrieval
ax2 = fig.add_subplot(3, 3, 2)
mn = list(pr_curves.keys()); mv = [pr_curves[n][2] * 100 for n in mn]
ax2.barh(mn, mv, color=['#E84040' if 'AMST' in n else '#5B7FA6' for n in mn])
ax2.set_title('(B) Retrieval MAP', fontsize=9); ax2.set_xlim(0, 105)
for iP, (nm_i, vi) in enumerate(zip(mn, mv)):
    ax2.text(vi + 0.5, iP, f'{vi:.1f}%', va='center', fontsize=8)

# (C) Accuracy vs Dimension
ax3 = fig.add_subplot(3, 3, 3)
for _, row in svm_df.iterrows():
    ia = 'AMST' in row['Method']
    ax3.scatter(row['Dim'], row['Accuracy'] * 100, s=150 if ia else 70,
                c='#E84040' if ia else '#5B7FA6', marker='s' if ia else 'o')
    ax3.annotate(row['Method'].split()[0], (row['Dim'], row['Accuracy'] * 100),
                 textcoords='offset points', xytext=(4, 2), fontsize=7)
ax3.set_xlabel('Descriptor Dimension', fontsize=9); ax3.set_ylabel('Accuracy (%)', fontsize=9)
ax3.set_title('(C) Accuracy vs Dimension', fontsize=9); ax3.grid(alpha=0.3)

# (D) Delta pp significance
ax4 = fig.add_subplot(3, 3, 4)
sig_df = stat_df.sort_values('Delta_pp', ascending=False)
colors_sig = ['#27AE60' if s else '#E74C3C' for s in sig_df['Significant']]
ax4.barh(sig_df['Baseline'], sig_df['Delta_pp'], color=colors_sig)
ax4.axvline(0, color='k', lw=1)
ax4.set_title('(D) AMST Gain over Baselines (pp)\nGreen=significant (p<0.05)', fontsize=9)
ax4.grid(axis='x', alpha=0.3); ax4.set_xlabel('Delta pp')

# (E) Summary text
ax5 = fig.add_subplot(3, 3, 5); ax5.axis('off')
summary_text = (
    f'=== v10 Summary ===\n'
    f'AMST (SVM):      {amst_acc_svm:.2f}%\n'
    f'AMST (Stacking): {amst_acc_stack:.2f}%\n'
    f'Best Baseline:   {best_base_svm:.2f}%\n'
    f'SVM Gain:        +{amst_acc_svm-best_base_svm:.2f} pp\n'
    f'Stk Gain:        +{amst_acc_stack-best_base_svm:.2f} pp\n'
    f'Sig wins:        {n_sig}/{len(stat_df)} (p<0.05)\n'
    f'Descriptor dim:  {AMST_DIM}'
)
ax5.text(0.05, 0.5, summary_text, fontsize=11, fontfamily='monospace', va='center',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# (F) Fixes applied
ax6 = fig.add_subplot(3, 3, 6); ax6.axis('off')
fixes_text = (
    'v10 Fixes Applied:\n'
    'FIX1: All methods use SVM-RBF\n'
    '      (fair comparison table)\n'
    'FIX2: Figs 8,9,13 y-axis = 0-100%\n'
    'FIX3: Dim = 466 everywhere\n'
    'FIX4: Ablation SVM for all rows;\n'
    '      stacking is a separate row'
)
ax6.text(0.05, 0.5, fixes_text, fontsize=10, fontfamily='monospace', va='center',
         bbox=dict(boxstyle='round', facecolor='#d4edda', alpha=0.4))

# (G) Ablation
ax7 = fig.add_subplot(3, 3, 7)
abl_simple = abl_df[['Method', 'Accuracy']].copy(); abl_simple['Accuracy'] *= 100
ax7.barh(abl_simple['Method'], abl_simple['Accuracy'],
         color=['#AED6F1', '#5DADE2', '#2471A3', '#1A5276', '#E84040'])
ax7.set_title('(G) Ablation (SVM for sub-variants)', fontsize=9); ax7.set_xlim(0, 115)

# (H) Acc vs F1 scatter
ax8 = fig.add_subplot(3, 3, 8)
for _, row in svm_df.iterrows():
    c = '#E84040' if 'AMST' in row['Method'] else '#5B7FA6'
    ax8.scatter(row['Accuracy'] * 100, row['F1_macro'] * 100, s=100, c=c, alpha=0.8)
ax8.set_xlabel('Accuracy (%)', fontsize=9); ax8.set_ylabel('F1 Macro (%)', fontsize=9)
ax8.set_title('(H) Acc vs F1', fontsize=9); ax8.grid(alpha=0.3)

# (I) Noise robustness AUC bar
ax9 = fig.add_subplot(3, 3, 9)
noise_auc = {nm: np.trapz([v * 100 for v in noise_res[nm]], noise_levels)
             for _, nm in noise_methods}
ax9.barh(list(noise_auc.keys()), list(noise_auc.values()),
         color=['#E84040' if 'AMST' in nm else '#5B7FA6' for nm in noise_auc])
ax9.set_title('(I) Noise Robustness AUC\n(area under accuracy-vs-noise curve)', fontsize=9)
ax9.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/fig12_dashboard.png', dpi=150, bbox_inches='tight')
plt.show(); print('Figure 12 saved.')

In [ ]:
# ── Final Results Summary ──────────────────────────────
print('=' * 80)
print('AMST v10 SHAPE DESCRIPTOR — FINAL RESULTS')
print('Dataset: KIMIA-216 | 18 classes x 12 = 216 images')
print('Classifier: SVM-RBF with GridSearchCV (ALL methods, fair comparison)')
print('Evaluation: 5-fold Stratified CV | Attention fitted per-fold (no leakage)')
print(f'Descriptor: {AMST_DIM}-dim (C1=144, C2=90, C3=210, C5=22)')
print('=' * 80)
print()
print('--- Fair Comparison (SVM-RBF for all) ---')
print(svm_df[['Method', 'Accuracy', 'Accuracy_std', 'F1_macro', 'Dim']].to_string(index=False))
print()
print(f'AMST (SVM)   : {amst_acc_svm:.2f}%  (descriptor gain)')
print(f'Best Baseline: {best_base_svm:.2f}%')
print(f'Descriptor gain: +{amst_acc_svm - best_base_svm:.2f} pp')
print()
print(f'--- Additional Stacking Experiment (AMST only) ---')
print(f'AMST (Stacking): {amst_acc_stack:.2f}%  (descriptor + classifier gain)')
print(f'Stacking bonus over SVM: +{amst_acc_stack - amst_acc_svm:.2f} pp')
print()
print(f'Significance: AMST outperforms {n_sig}/{len(stat_df)} baselines (p<0.05)')
print()
if amst_acc_svm > best_base_svm:
    print('SUCCESS: AMST descriptor alone (SVM) beats all baselines.')
else:
    print('NOTE: Descriptor improvement present but SVM parity not yet reached.')

In [ ]:
# ── Save All Results ──────────────────────────────
import os
out_dir = '/content/amst_v10_results'
os.makedirs(out_dir, exist_ok=True)

svm_df[['Method','Accuracy','Accuracy_std','F1_macro','Precision_macro','Recall_macro','Dim']].to_csv(
    f'{out_dir}/amst_classification_results_svm.csv', index=False)
stat_df.to_csv(f'{out_dir}/amst_significance_tests.csv', index=False)
pd.DataFrame(abl_svm_results).to_csv(f'{out_dir}/amst_ablation_svm.csv', index=False)
pd.DataFrame([{'Method': nm, 'MAP': v[2]} for nm, v in pr_curves.items()]).to_csv(
    f'{out_dir}/amst_map_scores.csv', index=False)
pd.DataFrame(noise_res, index=noise_levels).to_csv(f'{out_dir}/amst_noise_robustness.csv')
pd.DataFrame(occ_res, index=occ_levels).to_csv(f'{out_dir}/amst_occlusion_robustness.csv')
pd.DataFrame(rot_res, index=angles).to_csv(f'{out_dir}/amst_rotation_robustness.csv')

print('Saved files:')
for f in sorted(os.listdir(out_dir)):
    fp = os.path.join(out_dir, f)
    print(f'  {f:55s} {os.path.getsize(fp)//1024} KB')

print()
print('=== AMST v10 Complete — 13 Figures + 7 CSVs ===')
print('=== Fair comparison | No leakage | Correct axes | Consistent dims ===')

## References

1. Loncaric, S. (1998). A survey of shape analysis techniques. *Pattern Recognition*, 31(8), 983-1001.
2. Zhang, D., & Lu, G. (2004). Review of shape representation and description techniques. *Pattern Recognition*, 37(1), 1-19.
3. Belongie, S., Malik, J., & Puzicha, J. (2002). Shape matching and object recognition using shape contexts. *IEEE TPAMI*, 24(4), 509-522.
4. Carlsson, G. (2009). Topology and data. *Bulletin of the AMS*, 46(2), 255-308.
5. Edelsbrunner, H., & Harer, J. (2010). *Computational Topology: An Introduction*. AMS.
6. Barachant, A., et al. (2013). Classification of covariance matrices using a Riemannian kernel. *NeuroImage*, 75, 124-132.
7. Fisher, R. A. (1936). The use of multiple measurements in taxonomic problems. *Annals of Eugenics*, 7(2), 179-188.
8. Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5(2), 241-259.